In [1]:
import pandas as pd
import numpy as np

acs_need = pd.read_csv("Agg Data FINAL/Panhandle_Community_Need_Index.csv")
tracts = pd.read_csv("Agg Data FINAL/Panhandle_Census_Tracts.csv")
facilities = pd.read_excel("Agg Data FINAL/Panhandle_All_Facilities.xlsx")

print(acs_need.shape)
print(tracts.shape)
print(facilities.shape)

(27, 12)
(27, 4)
(17, 7)


In [2]:
print(tracts.columns)
print(facilities.columns)

Index(['County', 'Tract_GEOID', 'Latitude', 'Longitude'], dtype='object')
Index(['Facility_Name', 'County', 'City', 'Address', 'Longitude', 'Latitude',
       'Facility_Type'],
      dtype='object')


Note that these are straight line distances!

In [3]:

import numpy as np
from math import radians, sin, cos, sqrt, atan2

# calculate distance between two locations in miles
def calculate_distance(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1-a))

    return 3958.8 * c

# create a facility list without Sand Canyon
facilities_without = facilities[
    facilities["Facility_Name"] != "Sand Canyon Health Center"]

# find the nearest facility for each tract
nearest_with = []
nearest_without = []

for i, tract in tracts.iterrows():
    distances_with = []
    distances_without = []

    # distances with Sand Canyon
    for j, facility in facilities.iterrows():
        distance = calculate_distance(
            tract["Latitude"],
            tract["Longitude"],
            facility["Latitude"],
            facility["Longitude"])
        distances_with.append(distance)

    # distances without Sand Canyon
    for j, facility in facilities_without.iterrows():
        distance = calculate_distance(
            tract["Latitude"],
            tract["Longitude"],
            facility["Latitude"],
            facility["Longitude"])
        distances_without.append(distance)

    nearest_with.append(min(distances_with))
    nearest_without.append(min(distances_without))

# add distances to the tract data
tracts["Distance_With_Sand_Canyon"] = nearest_with
tracts["Distance_Without_Sand_Canyon"] = nearest_without

# calculate the difference between scenarios
tracts["Distance_Increase"] = (
    tracts["Distance_Without_Sand_Canyon"] - tracts["Distance_With_Sand_Canyon"])

display(tracts[[
    "County",
    "Tract_GEOID",
    "Distance_With_Sand_Canyon",
    "Distance_Without_Sand_Canyon",
    "Distance_Increase"]])

,County,Tract_GEOID,Distance_With_Sand_Canyon,Distance_Without_Sand_Canyon,Distance_Increase
0,BANNER,31007954000,21.233244,21.233244,0.0
1,BOX BUTTE,31013951100,5.860447,5.860447,0.0
2,BOX BUTTE,31013951200,5.654517,5.654517,0.0
3,BOX BUTTE,31013951300,5.731869,5.731869,0.0
4,CHEYENNE,31033954800,6.236515,6.236515,0.0
5,CHEYENNE,31033954900,2.576288,2.576288,0.0
6,CHEYENNE,31033955000,0.565491,0.565491,0.0
7,DAWES,31045950600,10.573570,10.573570,0.0
8,DAWES,31045950700,2.324525,2.324525,0.0
9,DEUEL,31049955400,13.819561,13.819561,0.0


In [4]:
tracts.to_csv("Agg Data FINAL/Panhandle_Tract_Facility_Distances.csv",index=False)

In [5]:
display(facilities[facilities["Facility_Name"] == "Sand Canyon Health Center"])

,Facility_Name,County,City,Address,Longitude,Latitude,Facility_Type
16,Sand Canyon Health Center,BOX BUTTE,HEMINGFORD,"800 Box Butte Ave, Hemingford, NE 69348",-103.073091,42.32187,Sand Canyon


In [6]:
for i, tract in tracts[tracts["County"] == "BOX BUTTE"].iterrows():

    distance = calculate_distance(
        tract["Latitude"],
        tract["Longitude"],
        42.32187009111592,
        -103.07309104622846
    )

    print(tract["Tract_GEOID"], distance)

31013951100 5.9455775039485355
31013951200 20.105519243234617
31013951300 23.344397451788527


In [7]:
print(tracts[[
    "County",
    "Tract_GEOID",
    "Distance_With_Sand_Canyon",
    "Distance_Without_Sand_Canyon",
    "Distance_Increase"]].query("County == 'BOX BUTTE'"))

      County  Tract_GEOID  Distance_With_Sand_Canyon  \
1  BOX BUTTE  31013951100                   5.860447   
2  BOX BUTTE  31013951200                   5.654517   
3  BOX BUTTE  31013951300                   5.731869   

   Distance_Without_Sand_Canyon  Distance_Increase  
1                      5.860447                0.0  
2                      5.654517                0.0  
3                      5.731869                0.0  


In [8]:
# check whether Sand Canyon is the nearest facility to any tract

for i, tract in tracts.iterrows():

    distance_to_sand_canyon = calculate_distance(
        tract["Latitude"],
        tract["Longitude"],
        42.32187009111592,
        -103.07309104622846
    )

    if distance_to_sand_canyon < tract["Distance_With_Sand_Canyon"]:
        print(tract["County"], tract["Tract_GEOID"],
              round(distance_to_sand_canyon, 2), "miles")

In [9]:

# create matching tract descriptions using the GEOIDs
tracts["Tract_Number"] = (tracts["Tract_GEOID"].astype(str).str[-6:].astype(int) // 100)
tracts["Tract"] = ("Census Tract " + tracts["Tract_Number"].astype(str) + "; " + tracts["County"].str.title() + " County; Nebraska")

# combine community need and facility distances
project_data = acs_need.merge(tracts[[
        "Tract",
        "Tract_GEOID",
        "Distance_With_Sand_Canyon",
        "Distance_Without_Sand_Canyon",
        "Distance_Increase"]],
    on="Tract",
    how="left")

print(project_data.shape)
print(project_data["Distance_With_Sand_Canyon"].isna().sum())
print(project_data["Distance_Without_Sand_Canyon"].isna().sum())

display(project_data[[
    "Tract",
    "Community_Need_Index",
    "Distance_With_Sand_Canyon",
    "Distance_Without_Sand_Canyon",
    "Distance_Increase"
]])


(27, 16)
0
0


,Tract,Community_Need_Index,Distance_With_Sand_Canyon,Distance_Without_Sand_Canyon,Distance_Increase
0,Census Tract 9545; Kimball County; Nebraska,0.877977,3.124918,3.124918,0.0
1,Census Tract 9537; Scotts Bluff County; Nebraska,0.821437,1.430032,1.430032,0.0
2,Census Tract 9531; Scotts Bluff County; Nebraska,0.683019,3.248792,3.248792,0.0
3,Census Tract 9535; Scotts Bluff County; Nebraska,0.650663,1.637908,1.637908,0.0
4,Census Tract 9521; Garden County; Nebraska,0.534964,15.436605,15.436605,0.0
5,Census Tract 9516; Sheridan County; Nebraska,0.422660,2.281271,2.281271,0.0
6,Census Tract 9517; Sheridan County; Nebraska,0.182884,18.919640,18.919640,0.0
7,Census Tract 9538; Scotts Bluff County; Nebraska,0.165542,2.255951,2.255951,0.0
8,Census Tract 9554; Deuel County; Nebraska,0.086290,13.819561,13.819561,0.0
9,Census Tract 9550; Cheyenne County; Nebraska,0.019821,0.565491,0.565491,0.0


In [10]:
project_data.to_csv("Agg Data FINAL/Panhandle_Need_and_Access.csv",index=False)

In [11]:
primary_care = pd.ExcelFile("Agg Data FINAL/Final_HRSA_Primary_Care.xlsx")
mental_health = pd.ExcelFile("Agg Data FINAL/Final HRSA Mental Health.xlsx")

print("Primary Care Sheets:", primary_care.sheet_names)
print("Mental Health Sheets:", mental_health.sheet_names)

Primary Care Sheets: ['SHORTAGE_AREA_HP_DETAIL_VX', 'Data Dictionary', 'Designation_Details', 'County_Summary', 'Read_Me']
Mental Health Sheets: ['SHORTAGE_AREA_HP_DETAIL_VX', 'Data Dictionary', 'Designation_Details', 'County_Summary', 'Read_Me']


In [12]:
#loading the county summary and designation details
pc_county = pd.read_excel("Agg Data FINAL/Final_HRSA_Primary_Care.xlsx",sheet_name="County_Summary")
mh_county = pd.read_excel("Agg Data FINAL/Final HRSA Mental Health.xlsx",sheet_name="County_Summary")
pc_details = pd.read_excel("Agg Data FINAL/Final_HRSA_Primary_Care.xlsx",sheet_name="Designation_Details")
mh_details = pd.read_excel("Agg Data FINAL/Final HRSA Mental Health.xlsx",sheet_name="Designation_Details")

# check the columns and first few rows
print("PRIMARY CARE")
print(pc_county.columns.tolist())
display(pc_county.head())

print("MENTAL HEALTH")
print(mh_county.columns.tolist())
display(mh_county.head())

PRIMARY CARE
['County', 'Active unique HPSAs', 'Active geographic', 'Active population', 'Active facility', 'Withdrawn unique HPSAs', 'Interpretation note']


,County,Active unique HPSAs,Active geographic,Active population,Active facility,Withdrawn unique HPSAs,Interpretation note
0,Banner,1,1,0,0,0,Designation applies to specified geography/pop...
1,Box Butte,1,0,1,0,1,Designation applies to specified geography/pop...
2,Cheyenne,2,0,1,1,0,Designation applies to specified geography/pop...
3,Dawes,2,0,1,1,2,Designation applies to specified geography/pop...
4,Deuel,1,1,0,0,3,Designation applies to specified geography/pop...


MENTAL HEALTH
['County', 'Designated HPSAs (unique IDs)', 'Withdrawn HPSAs (unique IDs)', 'All unique HPSA IDs', 'Source component records', 'Designation types (active)']


,County,Designated HPSAs (unique IDs),Withdrawn HPSAs (unique IDs),All unique HPSA IDs,Source component records,Designation types (active)
0,Banner,1,0,1,1,Geographic HPSA
1,Box Butte,1,0,1,1,Geographic HPSA
2,Cheyenne,2,0,2,2,Geographic HPSA; Rural Health Clinic
3,Dawes,2,1,3,3,Geographic HPSA; Rural Health Clinic
4,Deuel,1,0,1,1,Geographic HPSA


In [13]:
print("PRIMARY CARE DETAILS")
print(pc_details.columns.tolist())
display(pc_details.head(10))

print("MENTAL HEALTH DETAILS")
print(mh_details.columns.tolist())
display(mh_details.head(10))

PRIMARY CARE DETAILS
['County', 'HPSA Name', 'HPSA ID', 'Designation Type', 'Status', 'HPSA Score', 'Designation Date (Excel serial)', 'Last Update (Excel serial)', 'Component Name', 'Component Type', 'Population Type', 'Designation Population', 'Shortage', 'Provider FTE', 'Source County', 'Primary State', 'First occurrence of county + HPSA ID + status (1/0)', 'Category (Geographic / Population / Facility)', 'Original HRSA source row']


,County,HPSA Name,HPSA ID,Designation Type,Status,HPSA Score,Designation Date (Excel serial),Last Update (Excel serial),Component Name,Component Type,Population Type,Designation Population,Shortage,Provider FTE,Source County,Primary State,First occurrence of county + HPSA ID + status (1/0),Category (Geographic / Population / Facility),Original HRSA source row
0,Sheridan,GORDON CLINIC,13199931B4,Rural Health Clinic,Designated,18,41080,45754,GORDON CLINIC,Unknown,Rural Health Clinic,15718.0,NaN,NaN,"Sheridan County, NE",NE,1,Facility,34168
1,Kimball,KIMBALL COUNTY HOSPITAL CLINIC,13199931B3,Rural Health Clinic,Withdrawn,16,41080,43867,KIMBALL COUNTY HOSPITAL CLINIC,Unknown,Rural Health Clinic,5476.0,NaN,NaN,"Kimball County, NE",NE,1,Facility,34169
2,Deuel,Low Income-Sedgwick/Deuel(CO/NE),13199931B2,HPSA Population,Withdrawn,14,40899,40899,Deuel,Single County,Low Income Population HPSA,1876.0,0.33,0.3,"Deuel County, NE",NE,1,Population,34171
3,Sioux,Crawford,1319993103,Geographic HPSA,Withdrawn,0,30411,41089,Bowen,County Subdivision,Geographic Population,3729.0,0.90,0.2,"Sioux County, NE",NE,1,Geographic,34263
4,Dawes,Crawford,1319993103,Geographic HPSA,Withdrawn,0,30411,41089,07,County Subdivision,Geographic Population,3729.0,0.90,0.2,"Dawes County, NE",NE,1,Geographic,34264
5,Sioux,Crawford,1319993103,Geographic HPSA,Withdrawn,0,30411,41089,Whistle Creek,County Subdivision,Geographic Population,3729.0,0.90,0.2,"Sioux County, NE",NE,0,Geographic,34265
6,Dawes,Crawford,1319993103,Geographic HPSA,Withdrawn,0,30411,41089,09,County Subdivision,Geographic Population,3729.0,0.90,0.2,"Dawes County, NE",NE,0,Geographic,34266
7,Sioux,Crawford,1319993103,Geographic HPSA,Withdrawn,0,30411,41089,Sheep Creek,County Subdivision,Geographic Population,3729.0,0.90,0.2,"Sioux County, NE",NE,0,Geographic,34267
8,Dawes,Crawford,1319993103,Geographic HPSA,Withdrawn,0,30411,41089,10,County Subdivision,Geographic Population,3729.0,0.90,0.2,"Dawes County, NE",NE,0,Geographic,34268
9,Dawes,Crawford,1319993103,Geographic HPSA,Withdrawn,0,30411,41089,11,County Subdivision,Geographic Population,3729.0,0.90,0.2,"Dawes County, NE",NE,0,Geographic,34269


MENTAL HEALTH DETAILS
['County', 'HPSA Name', 'HPSA ID', 'Designation Type (HRSA)', 'HPSA Status', 'HPSA Score', 'HPSA Component Name', 'Component Type', 'Common County Name', 'County FIPS', 'Designation Date', 'Last Update Date', 'Withdrawn Date', 'Designation Population', 'Estimated Underserved Population', 'HPSA Shortage', 'Unique County/ID/Status Flag']


,County,HPSA Name,HPSA ID,Designation Type (HRSA),HPSA Status,HPSA Score,HPSA Component Name,Component Type,Common County Name,County FIPS,Designation Date,Last Update Date,Withdrawn Date,Designation Population,Estimated Underserved Population,HPSA Shortage,Unique County/ID/Status Flag
0,Banner,Region 1 MH Catchment Area,7318003632,Geographic HPSA,Designated,19,Banner,Single County,"Banner County, NE",31007,1978-07-21,2025-09-04,NaT,80231,50231.0,3.01,1
1,Box Butte,Region 1 MH Catchment Area,7318003632,Geographic HPSA,Designated,19,Box Butte,Single County,"Box Butte County, NE",31013,1978-07-21,2025-09-04,NaT,80231,50231.0,3.01,1
2,Cheyenne,Region 1 MH Catchment Area,7318003632,Geographic HPSA,Designated,19,Cheyenne,Single County,"Cheyenne County, NE",31033,1978-07-21,2025-09-04,NaT,80231,50231.0,3.01,1
3,Cheyenne,SIDNEY REGIONAL MEDICAL CENTER PHYSICIANS CLINIC,7319340118,Rural Health Clinic,Designated,16,SIDNEY REGIONAL MEDICAL CENTER PHYSICIANS CLINIC,Unknown,"Cheyenne County, NE",31033,2022-09-23,2025-04-07,NaT,13046,NaN,NaN,1
4,Dawes,LEGEND BUTTES HEALTH SERVICES-,7313245335,Rural Health Clinic,Designated,12,LEGEND BUTTES HEALTH SERVICES-,Unknown,"Dawes County, NE",31045,2021-09-30,2021-09-30,NaT,4990,NaN,NaN,1
5,Dawes,Region 1 MH Catchment Area,7318003632,Geographic HPSA,Designated,19,Dawes,Single County,"Dawes County, NE",31045,1978-07-21,2025-09-04,NaT,80231,50231.0,3.01,1
6,Dawes,LEGEND BUTTES HEALTH SERVICES-,731999314Q,Rural Health Clinic,Withdrawn,12,LEGEND BUTTES HEALTH SERVICES-,Unknown,"Dawes County, NE",31045,2010-11-15,2020-02-06,2020-02-06,5322,NaN,NaN,1
7,Deuel,Region 1 MH Catchment Area,7318003632,Geographic HPSA,Designated,19,Deuel,Single County,"Deuel County, NE",31049,1978-07-21,2025-09-04,NaT,80231,50231.0,3.01,1
8,Garden,Region 1 MH Catchment Area,7318003632,Geographic HPSA,Designated,19,Garden,Single County,"Garden County, NE",31069,1978-07-21,2025-09-04,NaT,80231,50231.0,3.01,1
9,Garden,REGIONAL WEST GARDEN COUNTY CLINIC,7319993154,Rural Health Clinic,Withdrawn,13,REGIONAL WEST GARDEN COUNTY CLINIC,Unknown,"Garden County, NE",31069,2003-11-10,2020-02-06,2020-02-06,6139,NaN,NaN,1


In [14]:
# keeping only currently designated primary care HPSAs
pc_active = pc_details[pc_details["Status"] == "Designated"].copy()
# create separate indicators for each type of designation
pc_active["Geographic"] = (pc_active["Category (Geographic / Population / Facility)"] == "Geographic").astype(int)
pc_active["Population"] = (pc_active["Category (Geographic / Population / Facility)"] == "Population").astype(int)
pc_active["Facility"] = (pc_active["Category (Geographic / Population / Facility)"] == "Facility").astype(int)

# summarizing the indicators by county
pc_flags = pc_active.groupby("County")[["Geographic", "Population", "Facility"]].max().reset_index()

pc_flags = pc_flags.rename(columns={"Geographic": "PC_Geographic_HPSA","Population": "PC_Population_HPSA","Facility": "PC_Facility_HPSA"})
display(pc_flags)

,County,PC_Geographic_HPSA,PC_Population_HPSA,PC_Facility_HPSA
0,Banner,1,0,0
1,Box Butte,0,1,0
2,Cheyenne,0,1,1
3,Dawes,0,1,1
4,Deuel,1,0,0
5,Garden,1,0,0
6,Kimball,0,1,0
7,Morrill,0,1,1
8,Scotts Bluff,0,1,1
9,Sheridan,1,0,1


In [15]:
mh_active = mh_details[mh_details["HPSA Status"] == "Designated"].copy()
mh_active["Geographic"] = (mh_active["Designation Type (HRSA)"].str.contains("Geographic", case=False)).astype(int)
mh_active["Population"] = (mh_active["Designation Type (HRSA)"].str.contains("Population", case=False)).astype(int)
mh_active["Facility"] = (mh_active["Designation Type (HRSA)"].str.contains("Rural Health Clinic", case=False, na=False)).astype(int)
mh_flags = mh_active.groupby("County")[["Geographic", "Population", "Facility"]].max().reset_index()

mh_flags = mh_flags.rename(columns={"Geographic": "MH_Geographic_HPSA","Population": "MH_Population_HPSA","Facility": "MH_Facility_HPSA"})
display(mh_flags)

,County,MH_Geographic_HPSA,MH_Population_HPSA,MH_Facility_HPSA
0,Banner,1,0,0
1,Box Butte,1,0,0
2,Cheyenne,1,0,1
3,Dawes,1,0,1
4,Deuel,1,0,0
5,Garden,1,0,0
6,Kimball,1,0,0
7,Morrill,1,0,1
8,Scotts Bluff,1,0,0
9,Sheridan,1,0,1


In [16]:
print(project_data.columns.tolist())

['Tract', 'Poverty_Rate', 'Uninsured_Rate', 'Age_65_Plus', 'Disability_Rate', 'SNAP_Rate', 'Poverty_Rate_Z', 'Uninsured_Rate_Z', 'Age_65_Plus_Z', 'Disability_Rate_Z', 'SNAP_Rate_Z', 'Community_Need_Index', 'Tract_GEOID', 'Distance_With_Sand_Canyon', 'Distance_Without_Sand_Canyon', 'Distance_Increase']


In [17]:
# adding county names to the tract data
county_lookup = tracts[["Tract_GEOID", "County"]]

project_data = project_data.merge(county_lookup,on="Tract_GEOID",how="left")

# standardizing county names before merging
project_data["County"] = project_data["County"].str.strip().str.title()
pc_flags["County"] = pc_flags["County"].str.strip().str.title()
mh_flags["County"] = mh_flags["County"].str.strip().str.title()

# merging the primary care and mental health indicators
project_data = project_data.merge(pc_flags, on="County", how="left")
project_data = project_data.merge(mh_flags, on="County", how="left")

# replacing missing indicators with zero
hpsa_columns = ["PC_Geographic_HPSA", "PC_Population_HPSA", "PC_Facility_HPSA","MH_Geographic_HPSA", "MH_Population_HPSA", "MH_Facility_HPSA"]
project_data[hpsa_columns] = project_data[hpsa_columns].fillna(0).astype(int)

display(project_data[["County","Community_Need_Index", "Distance_With_Sand_Canyon", "Distance_Without_Sand_Canyon", "Distance_Increase"] + hpsa_columns])

,County,Community_Need_Index,Distance_With_Sand_Canyon,Distance_Without_Sand_Canyon,Distance_Increase,PC_Geographic_HPSA,PC_Population_HPSA,PC_Facility_HPSA,MH_Geographic_HPSA,MH_Population_HPSA,MH_Facility_HPSA
0,Kimball,0.877977,3.124918,3.124918,0.0,0,1,0,1,0,0
1,Scotts Bluff,0.821437,1.430032,1.430032,0.0,0,1,1,1,0,0
2,Scotts Bluff,0.683019,3.248792,3.248792,0.0,0,1,1,1,0,0
3,Scotts Bluff,0.650663,1.637908,1.637908,0.0,0,1,1,1,0,0
4,Garden,0.534964,15.436605,15.436605,0.0,1,0,0,1,0,0
5,Sheridan,0.422660,2.281271,2.281271,0.0,1,0,1,1,0,1
6,Sheridan,0.182884,18.919640,18.919640,0.0,1,0,1,1,0,1
7,Scotts Bluff,0.165542,2.255951,2.255951,0.0,0,1,1,1,0,0
8,Deuel,0.086290,13.819561,13.819561,0.0,1,0,0,1,0,0
9,Cheyenne,0.019821,0.565491,0.565491,0.0,0,1,1,1,0,1


In [18]:

project_data.to_csv("Agg Data FINAL/Panhandle_Final_Project_Data.csv",index=False)

In [19]:
print(project_data.shape)
print(project_data.isna().sum())
print(project_data.columns.tolist())

(27, 23)
Tract                           0
Poverty_Rate                    0
Uninsured_Rate                  0
Age_65_Plus                     0
Disability_Rate                 0
SNAP_Rate                       0
Poverty_Rate_Z                  0
Uninsured_Rate_Z                0
Age_65_Plus_Z                   0
Disability_Rate_Z               0
SNAP_Rate_Z                     0
Community_Need_Index            0
Tract_GEOID                     0
Distance_With_Sand_Canyon       0
Distance_Without_Sand_Canyon    0
Distance_Increase               0
County                          0
PC_Geographic_HPSA              0
PC_Population_HPSA              0
PC_Facility_HPSA                0
MH_Geographic_HPSA              0
MH_Population_HPSA              0
MH_Facility_HPSA                0
dtype: int64
['Tract', 'Poverty_Rate', 'Uninsured_Rate', 'Age_65_Plus', 'Disability_Rate', 'SNAP_Rate', 'Poverty_Rate_Z', 'Uninsured_Rate_Z', 'Age_65_Plus_Z', 'Disability_Rate_Z', 'SNAP_Rate_Z', 'Community_Nee